In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


# -------------------------
# 1. Load Model
# -------------------------

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)


# -------------------------
# 2. Load Dataset
# -------------------------

df = pd.read_csv("/Users/kaustavkhanikar/Desktop/Working/Paper_3/FAQ/dataset.csv")  # columns: question, answer
df = df.dropna(subset=["question", "answer"])
dataset = Dataset.from_pandas(df)


# -------------------------
# 3. Preprocess (Manual Masking)
# -------------------------

def preprocess(example):
    # Build chat format properly
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Tokenize full sequence
    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=512,
    )

    # Now mask everything before assistant response
    # Find where assistant answer starts

    assistant_text = tokenizer.apply_chat_template(
        [{"role": "assistant", "content": example["answer"]}],
        tokenize=False,
        add_generation_prompt=False,
    )

    assistant_tokens = tokenizer(
        assistant_text,
        truncation=True,
        max_length=512,
        add_special_tokens=False,
    )["input_ids"]

    labels = tokenized["input_ids"].copy()

    # Mask all tokens except assistant answer
    labels[:-len(assistant_tokens)] = [-100] * (len(labels) - len(assistant_tokens))

    tokenized["labels"] = labels

    return tokenized


dataset = dataset.map(preprocess, remove_columns=dataset.column_names)


# -------------------------
# 4. LoRA Config
# -------------------------

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)


# -------------------------
# 5. SFTConfig
# -------------------------

training_args = SFTConfig(
    output_dir="./llama3.2-library-faq",
    per_device_train_batch_size=4,
    num_train_epochs=12,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    report_to="none",
    packing=False,
)



`torch_dtype` is deprecated! Use `dtype` instead!
Map: 100%|██████████| 116/116 [00:00<00:00, 2623.19 examples/s]


In [ ]:

# -------------------------
# 6. Trainer
# -------------------------

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
)


# -------------------------
# 7. Train
# -------------------------

trainer.train()

trainer.model.save_pretrained("./llama3.2-library-faq-adapter")


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


# -------------------------
# 1. Load Base Model
# -------------------------

base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "./llama3.2-library-faq-adapter"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# -------------------------
# 2. Load LoRA Adapter
# -------------------------

model = PeftModel.from_pretrained(base_model, adapter_path)

model.eval()



Loading weights: 100%|██████████| 146/146 [00:04<00:00, 33.64it/s, Materializing param=model.norm.weight]                              


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [21]:

# -------------------------
# 3. Ask Question
# -------------------------

question = "Can I borrow reference books?"

messages = [
    {"role": "user", "content": question},
]

# Build chat prompt
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
).to(model.device)


# -------------------------
# 4. Generate Response
# -------------------------

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("\nAssistant:", response)



Assistant: Sorry, but reference books are very rare and you don't have to borrow them.
